In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/raw/cs-training.csv")

df = df.drop(columns=["Unnamed: 0"])
df = df[df["age"] > 0]

df = df[
    ~df["NumberOfTimes90DaysLate"].isin([96, 98])
]

df.shape

(149730, 11)

In [3]:
X = df.drop(columns=["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
train_woe = X_train.copy()
train_woe["target"] = y_train

In [5]:
train_woe["age_bin"], age_bins = pd.qcut(
    train_woe["age"],
    q=10,
    retbins=True,
    duplicates="drop"
)

age_bins

array([ 21.,  33.,  39.,  44.,  48.,  52.,  56.,  61.,  65.,  72., 109.])

In [6]:
age_woe = (
    train_woe.groupby("age_bin", observed=False)["target"]
    .agg(["count", "sum"])
)

age_woe

,count,sum
age_bin,,
"(20.999, 33.0]",13540,1493
"(33.0, 39.0]",11942,1131
"(39.0, 44.0]",12580,1053
"(44.0, 48.0]",11742,941
"(48.0, 52.0]",11892,925
"(52.0, 56.0]",11388,752
"(56.0, 61.0]",13481,664
"(61.0, 65.0]",10369,398
"(65.0, 72.0]",11364,307


In [7]:
age_woe["bad"] = age_woe["sum"]
age_woe["good"] = age_woe["count"] - age_woe["bad"]

In [8]:
total_good = age_woe["good"].sum()
total_bad = age_woe["bad"].sum()

age_woe["dist_good"] = age_woe["good"] / total_good
age_woe["dist_bad"] = age_woe["bad"] / total_bad

In [9]:
epsilon = 1e-10

age_woe["woe"] = np.log(
    (age_woe["dist_good"] + epsilon) /
    (age_woe["dist_bad"] + epsilon)
)

In [10]:
age_woe[
    [
        "count",
        "good",
        "bad",
        "dist_good",
        "dist_bad",
        "woe"
    ]
]

,count,good,bad,dist_good,dist_bad,woe
age_bin,,,,,,
"(20.999, 33.0]",13540,12047,1493,0.107677,0.188916,-0.562165
"(33.0, 39.0]",11942,10811,1131,0.096629,0.143110,-0.392731
"(39.0, 44.0]",12580,11527,1053,0.103029,0.133241,-0.257144
"(44.0, 48.0]",11742,10801,941,0.096540,0.119069,-0.209743
"(48.0, 52.0]",11892,10967,925,0.098024,0.117044,-0.177341
"(52.0, 56.0]",11388,10636,752,0.095065,0.095154,-0.000930
"(56.0, 61.0]",13481,12817,664,0.114559,0.084019,0.310052
"(61.0, 65.0]",10369,9971,398,0.089121,0.050361,0.570791
"(65.0, 72.0]",11364,11057,307,0.098828,0.038846,0.933778


In [11]:
age_woe["iv_component"] = (
    age_woe["dist_good"] - age_woe["dist_bad"]
) * age_woe["woe"]

age_iv = age_woe["iv_component"].sum()

print("Age Information Value:", age_iv)

Age Information Value: 0.2518207116333622


# Weight of Evidence & Information Value

Weight of Evidence (WOE) measures how strongly each group of a variable is associated with good versus bad borrowers.

Positive WOE values indicate a relatively higher concentration of non-default borrowers, while negative WOE values indicate a higher concentration of defaults.

Information Value (IV) summarizes the discriminatory power of the variable across all bins.

All binning and WOE calculations are derived exclusively from the training sample to avoid data leakage.

In [12]:
print("Age Information Value:", age_iv)

Age Information Value: 0.2518207116333622


In [13]:
def calculate_iv(data, feature, target, bins=10):
    temp = data[[feature, target]].copy()

    # Treat missing values as a separate group
    missing_mask = temp[feature].isnull()

    non_missing = temp.loc[~missing_mask].copy()
    missing = temp.loc[missing_mask].copy()

    # Bin continuous variables
    if non_missing[feature].nunique() > bins:
        try:
            non_missing["bin"] = pd.qcut(
                non_missing[feature],
                q=bins,
                duplicates="drop"
            )
        except ValueError:
            non_missing["bin"] = non_missing[feature]
    else:
        non_missing["bin"] = non_missing[feature].astype(str)

    # Convert bins to string so missing can be added
    non_missing["bin"] = non_missing["bin"].astype(str)

    if len(missing) > 0:
        missing["bin"] = "Missing"
        temp_binned = pd.concat([non_missing, missing], axis=0)
    else:
        temp_binned = non_missing

    grouped = (
        temp_binned.groupby("bin")[target]
        .agg(["count", "sum"])
    )

    grouped["bad"] = grouped["sum"]
    grouped["good"] = grouped["count"] - grouped["bad"]

    total_good = grouped["good"].sum()
    total_bad = grouped["bad"].sum()

    grouped["dist_good"] = grouped["good"] / total_good
    grouped["dist_bad"] = grouped["bad"] / total_bad

    epsilon = 1e-10

    grouped["woe"] = np.log(
        (grouped["dist_good"] + epsilon) /
        (grouped["dist_bad"] + epsilon)
    )

    grouped["iv_component"] = (
        grouped["dist_good"] - grouped["dist_bad"]
    ) * grouped["woe"]

    iv = grouped["iv_component"].sum()

    return iv, grouped

In [14]:
train_iv = X_train.copy()
train_iv["target"] = y_train

In [15]:
iv_results = []

for feature in X_train.columns:
    iv, _ = calculate_iv(
        train_iv,
        feature,
        "target",
        bins=10
    )

    iv_results.append({
        "feature": feature,
        "IV": iv
    })

iv_table = pd.DataFrame(iv_results)

iv_table = iv_table.sort_values(
    by="IV",
    ascending=False
)

iv_table

,feature,IV
0,RevolvingUtilizationOfUnsecuredLines,1.105271
8,NumberOfTime60-89DaysPastDueNotWorse,0.572044
2,NumberOfTime30-59DaysPastDueNotWorse,0.451675
1,age,0.251821
3,DebtRatio,0.083496
4,MonthlyIncome,0.071582
5,NumberOfOpenCreditLinesAndLoans,0.053323
9,NumberOfDependents,0.027275
7,NumberRealEstateLoansOrLines,0.011013
6,NumberOfTimes90DaysLate,0.000000


In [16]:
df["NumberOfTimes90DaysLate"].value_counts().sort_index()

NumberOfTimes90DaysLate
0     141661
1       5243
2       1555
3        667
4        291
5        131
6         80
7         38
8         21
9         19
10         8
11         5
12         2
13         4
14         2
15         2
17         1
Name: count, dtype: int64

In [17]:
late90_iv_data = train_iv[
    ["NumberOfTimes90DaysLate", "target"]
].copy()

late90_iv_data["bin"] = late90_iv_data[
    "NumberOfTimes90DaysLate"
].astype(str)

late90_grouped = (
    late90_iv_data.groupby("bin")["target"]
    .agg(["count", "sum"])
)

late90_grouped["bad"] = late90_grouped["sum"]
late90_grouped["good"] = (
    late90_grouped["count"] - late90_grouped["bad"]
)

total_good = late90_grouped["good"].sum()
total_bad = late90_grouped["bad"].sum()

late90_grouped["dist_good"] = (
    late90_grouped["good"] / total_good
)

late90_grouped["dist_bad"] = (
    late90_grouped["bad"] / total_bad
)

epsilon = 1e-10

late90_grouped["woe"] = np.log(
    (late90_grouped["dist_good"] + epsilon) /
    (late90_grouped["dist_bad"] + epsilon)
)

late90_grouped["iv_component"] = (
    late90_grouped["dist_good"] -
    late90_grouped["dist_bad"]
) * late90_grouped["woe"]

late90_iv = late90_grouped["iv_component"].sum()

print("90+ Days Late IV:", late90_iv)

late90_grouped

90+ Days Late IV: 0.8563480589889451


,count,sum,bad,good,dist_good,dist_bad,woe,iv_component
bin,,,,,,,,
0,113307,5231,5231,108076,0.965991,0.661901,0.378039,0.114958
1,4186,1412,1412,2774,0.024794,0.178666,-1.974910,0.303884
10,6,4,4,2,0.000018,0.000506,-3.343335,0.001632
11,3,1,1,2,0.000018,0.000127,-1.957041,0.000213
12,2,1,1,1,0.000009,0.000127,-2.650183,0.000312
13,4,2,2,2,0.000018,0.000253,-2.650188,0.000623
15,2,0,0,2,0.000018,0.000000,12.093813,0.000216
17,1,1,1,0,0.000000,0.000127,-14.050854,0.001778
2,1251,622,622,629,0.005622,0.078704,-2.639002,0.192864


In [18]:
util_iv, util_table = calculate_iv(
    train_iv,
    "RevolvingUtilizationOfUnsecuredLines",
    "target",
    bins=10
)

print("IV:", util_iv)

util_table[
    [
        "count",
        "good",
        "bad",
        "dist_good",
        "dist_bad",
        "woe",
        "iv_component"
    ]
]

IV: 1.1052709405898427


,count,good,bad,dist_good,dist_bad,woe,iv_component
bin,,,,,,,
"(-0.001, 0.00294]",11979,11687,292,0.104459,0.036948,1.039285,0.070163
"(0.00294, 0.0192]",11978,11815,163,0.105603,0.020625,1.633182,0.138785
"(0.0192, 0.0433]",11978,11806,172,0.105523,0.021764,1.578675,0.132228
"(0.0433, 0.0824]",11979,11748,231,0.105004,0.029229,1.278827,0.096903
"(0.0824, 0.153]",11978,11694,284,0.104522,0.035936,1.067664,0.073227
"(0.153, 0.27]",11978,11569,409,0.103405,0.051752,0.692176,0.035752
"(0.27, 0.442]",11979,11369,610,0.101617,0.077186,0.274993,0.006718
"(0.442, 0.696]",11978,10953,1025,0.097899,0.129698,-0.281273,0.008944
"(0.696, 0.979]",11978,9978,2000,0.089184,0.253068,-1.042958,0.170925


In [19]:
late90_data = train_iv[
    ["NumberOfTimes90DaysLate", "target"]
].copy()

late90_data["late90_group"] = pd.cut(
    late90_data["NumberOfTimes90DaysLate"],
    bins=[-1, 0, 1, 2, 3, np.inf],
    labels=["0", "1", "2", "3", "4+"]
)

In [20]:
late90_data["late90_group"].value_counts().sort_index()

late90_group
0     113307
1       4186
2       1251
3        549
4+       491
Name: count, dtype: int64

In [21]:
late90_woe = (
    late90_data.groupby(
        "late90_group",
        observed=False
    )["target"]
    .agg(["count", "sum"])
)

late90_woe["bad"] = late90_woe["sum"]
late90_woe["good"] = late90_woe["count"] - late90_woe["bad"]

total_good = late90_woe["good"].sum()
total_bad = late90_woe["bad"].sum()

late90_woe["dist_good"] = late90_woe["good"] / total_good
late90_woe["dist_bad"] = late90_woe["bad"] / total_bad

epsilon = 1e-10

late90_woe["woe"] = np.log(
    (late90_woe["dist_good"] + epsilon) /
    (late90_woe["dist_bad"] + epsilon)
)

late90_woe["iv_component"] = (
    late90_woe["dist_good"] -
    late90_woe["dist_bad"]
) * late90_woe["woe"]

late90_iv = late90_woe["iv_component"].sum()

print("90+ Days Late IV:", late90_iv)

late90_woe

90+ Days Late IV: 0.8532176997584229


,count,sum,bad,good,dist_good,dist_bad,woe,iv_component
late90_group,,,,,,,,
0,113307,5231,5231,108076,0.965991,0.661901,0.378039,0.114958
1,4186,1412,1412,2774,0.024794,0.178666,-1.974910,0.303884
2,1251,622,622,629,0.005622,0.078704,-2.639002,0.192864
3,549,314,314,235,0.002100,0.039732,-2.940001,0.110636
4+,491,324,324,167,0.001493,0.040997,-3.312943,0.130876


In [22]:
util_iv, util_table = calculate_iv(
    train_iv,
    "RevolvingUtilizationOfUnsecuredLines",
    "target",
    bins=10
)

print("Utilization IV:", util_iv)

util_table[
    [
        "count",
        "good",
        "bad",
        "dist_good",
        "dist_bad",
        "woe",
        "iv_component"
    ]
]

Utilization IV: 1.1052709405898427


,count,good,bad,dist_good,dist_bad,woe,iv_component
bin,,,,,,,
"(-0.001, 0.00294]",11979,11687,292,0.104459,0.036948,1.039285,0.070163
"(0.00294, 0.0192]",11978,11815,163,0.105603,0.020625,1.633182,0.138785
"(0.0192, 0.0433]",11978,11806,172,0.105523,0.021764,1.578675,0.132228
"(0.0433, 0.0824]",11979,11748,231,0.105004,0.029229,1.278827,0.096903
"(0.0824, 0.153]",11978,11694,284,0.104522,0.035936,1.067664,0.073227
"(0.153, 0.27]",11978,11569,409,0.103405,0.051752,0.692176,0.035752
"(0.27, 0.442]",11979,11369,610,0.101617,0.077186,0.274993,0.006718
"(0.442, 0.696]",11978,10953,1025,0.097899,0.129698,-0.281273,0.008944
"(0.696, 0.979]",11978,9978,2000,0.089184,0.253068,-1.042958,0.170925


In [23]:
selected_features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "age",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

## Information Value Results

Information Value analysis identified several variables with strong discriminatory power.

The strongest predictors were:

- Revolving credit utilization
- 90+ day delinquency history
- 60-89 day delinquency history
- 30-59 day delinquency history

Age showed moderate predictive strength, while income, debt ratio and number of credit lines provided weaker but potentially useful information.

Variables with very high IV values were retained for further analysis but will be monitored carefully for instability, extreme values and possible overfitting.